# ⚡ RealityLoops Custom GPU Server (Stable Fast 3D)

This notebook configures and runs a dedicated private GPU server hosting **Stable Fast 3D** for the **Premium lite** generation mode of RealityLoops.

### ✅ T4 GPU Compatible
This version fixes the `bfloat16` crash on T4 GPUs by automatically detecting compute capability and using `float16` instead.

### 🚀 Steps:
1. **Run Cell 1** to verify GPU and clone the Stable Fast 3D repository.
2. **Run Cell 2** to install all required dependencies (including PyNanoInstantMeshes, uv_unwrapper, texture_baker, and pinned transformers).
3. **Run Cell 3** to log in to Hugging Face using your access token.
4. **Run Cell 4** to write the T4-compatible FastAPI server script.
5. **Run Cell 5**, enter your **ngrok authtoken**, and execute to start the public tunnel and server.

### ⚙️ Step 1: Verify GPU & Clone Repository

In [ ]:
# Verify GPU availability (Make sure T4 or other GPU is active)
!nvidia-smi

# Clone the Stable Fast 3D repository
!git clone https://github.com/Stability-AI/stable-fast-3d.git
%cd stable-fast-3d

### 📦 Step 2: Install All Dependencies (Consolidated)

In [ ]:
# Install all required dependencies at once to avoid conflicts or missing packages
!pip install onnxruntime gpytoolbox "rembg[gpu]" open-clip-torch einops trimesh omegaconf jaxtyping fastapi uvicorn pyngrok nest-asyncio python-multipart diffusers fire pynanoinstantmeshes Pillow -q

# Install local submodules with no build isolation (to prevent PyTorch build issues)
!pip install ./uv_unwrapper --no-build-isolation
!pip install ./texture_baker --no-build-isolation

# Downgrade transformers to a version compatible with Stable Fast 3D
!pip install transformers==4.40.2 -q

# Install Stable Fast 3D package itself without reinstalling dependencies
!pip install --no-deps .

print("✅ All dependencies installed successfully!")

### 🔑 Step 3: Hugging Face Login (For Model Weight Access)

In [ ]:
from huggingface_hub import login

# Get a free Read token from https://huggingface.co/settings/tokens
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"

if HF_TOKEN == "PASTE_YOUR_HF_TOKEN_HERE" or not HF_TOKEN:
    print("⚠️ WARNING: Please paste your Hugging Face Token inside the HF_TOKEN variable!")
else:
    login(HF_TOKEN)

### 🖥️ Step 4: Write T4-Compatible FastAPI Server Script

**Key fix:** The original `run.py` uses `bfloat16` precision which crashes on T4 GPUs (Turing architecture).
This server detects your GPU's compute capability and uses `float16` on T4, `bfloat16` on A100/newer.

In [ ]:
%%writefile server.py
import os
import gc
# Set PyTorch memory allocator config to prevent fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import shutil
import torch
import rembg
from contextlib import nullcontext
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image, ImageOps

from sf3d.system import SF3D
from sf3d.utils import get_device, remove_background, resize_foreground

app = FastAPI(title="RealityLoops Custom GPU Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Detect device and pick correct dtype ──────────────────────────
device = get_device()
if "cuda" in device:
    # T4 (compute capability 7.5 / Turing) does NOT support bfloat16.
    # Only Ampere+ (compute capability >= 8.0) supports it.
    capability = torch.cuda.get_device_capability()
    if capability[0] >= 8:
        autocast_dtype = torch.bfloat16
        print(f"GPU supports bfloat16 (cc {capability[0]}.{capability[1]}), using bfloat16")
    else:
        autocast_dtype = torch.float16
        print(f"GPU does NOT support bfloat16 (cc {capability[0]}.{capability[1]}), using float16")
else:
    autocast_dtype = torch.float32
    print("No CUDA device, using float32")

# ── Load model once at startup ────────────────────────────────────
print("Loading Stable Fast 3D model...")
model = SF3D.from_pretrained(
    "stabilityai/stable-fast-3d",
    config_name="config.yaml",
    weight_name="model.safetensors",
)
model.to(device)
model.eval()
rembg_session = rembg.new_session()
print("Model loaded and ready!")


@app.post("/generate-3d")
async def generate_3d(file: UploadFile = File(...)):
    temp_dir = "temp_run"
    os.makedirs(temp_dir, exist_ok=True)

    # Save and resize uploaded image (max 512x512 to prevent CUDA OOM)
    image_path = os.path.join(temp_dir, "input.png")
    try:
        with Image.open(file.file) as img:
            img = ImageOps.exif_transpose(img)
            img.thumbnail((512, 512))
            img.save(image_path, "PNG")
    except Exception as e:
        print(f"Failed to process/resize image: {e}")
        file.file.seek(0)
        with open(image_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)

    output_dir = os.path.join(temp_dir, "output")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    try:
        print("Running Stable Fast 3D inference (in-process, T4-safe)...")
        input_image = Image.open(image_path)
        input_image = remove_background(input_image, rembg_session)
        input_image = resize_foreground(input_image, 0.85)

        with torch.no_grad():
            with torch.autocast(
                device_type=device, dtype=autocast_dtype
            ) if "cuda" in device else nullcontext():
                mesh, glob_dict = model.run_image(
                    input_image,
                    bake_resolution=1024,
                    remesh="none",
                )

        out_path = os.path.join(output_dir, "mesh.glb")
        mesh.export(out_path, include_normals=True)
        print(f"GLB saved to {out_path}")

        del mesh, glob_dict
        gc.collect()
        torch.cuda.empty_cache()

        return FileResponse(out_path, media_type="model/gltf-binary", filename="model.glb")

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        print(f"Generation failed:\n{tb}")
        gc.collect()
        torch.cuda.empty_cache()
        return {"error": "Generation failed", "details": str(e)}

### 🚀 Step 5: Configure Ngrok and Start Server

In [ ]:
# @title Configure and Start Tunnel
# @markdown Enter your ngrok authtoken below (obtain a free token from https://dashboard.ngrok.com)
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"  # @param {type:"string"}

import nest_asyncio
from pyngrok import ngrok
import uvicorn

if not NGROK_TOKEN or NGROK_TOKEN == "PASTE_YOUR_NGROK_TOKEN_HERE":
    print("⚠️ WARNING: NGROK_TOKEN is empty! Please paste a valid token from https://dashboard.ngrok.com")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    try:
        # Expose port 8000 using ngrok tunnel
        public_url = ngrok.connect(8000).public_url
        
        print("\n" + "="*60)
        print("🎉 SUCCESS: Your Custom GPU Server is live!")
        print(f"👉 COPY THIS URL: {public_url}")
        print("👉 Open your local backend .env file and update:")
        print(f"   CUSTOM_GPU_URL={public_url}")
        print("="*60 + "\n")
        
        # Apply nest_asyncio to support running uvicorn in Jupyter
        nest_asyncio.apply()
        
        # Start the FastAPI server safely inside Jupyter event loop
        config = uvicorn.Config("server:app", host="0.0.0.0", port=8000, log_level="info")
        server = uvicorn.Server(config)
        await server.serve()
    except Exception as e:
        print(f"❌ Error starting tunnel: {e}")